<a href="https://colab.research.google.com/github/L-Poca/Data_Pipeline/blob/rafael_cleaning/notebooks/colab/transfer_learning_refactored.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Transfer Learning - Version Refactorisée

Ce notebook utilise les fonctions utilitaires de `src.notebooks.notebook_utils` pour le Transfer Learning.

**Modèles testés:**
- InceptionV3 (recommandé)
- VGG16
- ResNet50
- EfficientNetB0

**Stratégie 2-phases:**
1. **Feature Extraction** - Base model gelée
2. **Fine-Tuning** - Dégeler les top layers

## 1. Configuration et Imports

In [ ]:
"""
╔════════════════════════════════════════════════════════════════════════════╗
║  🎯 CELLULE DE CONFIGURATION STANDALONE - COPIER-COLLER DANS VOS NOTEBOOKS ║
╚════════════════════════════════════════════════════════════════════════════╝

INSTRUCTIONS:
-------------
1. Copiez TOUT le contenu de cette cellule
2. Collez-le comme PREMIÈRE CELLULE de votre notebook
3. Exécutez la cellule
4. Les variables sont prêtes à l'emploi !

Cette cellule est 100% autonome et fonctionne partout :
✅ Google Colab (clone + installe automatiquement)
✅ WSL / Linux Local
✅ Tout environnement Jupyter

APRÈS EXÉCUTION, VOUS POUVEZ UTILISER:
- config: Objet de configuration (config.batch_size, config.data_dir, etc.)
- ENV: Environnement détecté ('colab', 'wsl', 'local')
- Tous les imports des transformers

"""

# =============================================================================
# IMPORTS STANDARDS
# =============================================================================

import os
import sys
import subprocess
from pathlib import Path


# =============================================================================
# DÉTECTION AUTOMATIQUE DE L'ENVIRONNEMENT
# =============================================================================

def detect_environment():
    """Détecte l'environnement (colab, wsl, local)"""
    try:
        import google.colab
        return "colab"
    except ImportError:
        is_wsl = os.path.exists('/proc/version') and 'microsoft' in open('/proc/version').read().lower()
        return "wsl" if is_wsl else "local"

ENV = detect_environment()
print(f"🌍 Environnement: {ENV.upper()}")


# =============================================================================
# BOOTSTRAP COLAB (Clone + Install si nécessaire)
# =============================================================================

if ENV == "colab":
    print("\n🚀 Bootstrap Colab...")
    
    os.chdir('/content')
    if not os.path.exists('/content/Data_Pipeline'):
        print("📥 Clonage du repository...")
        subprocess.run(['git', 'clone', 'https://github.com/L-Poca/Data_Pipeline.git'], check=True)
    
    os.chdir('/content/Data_Pipeline')
    
    # Checkout de la branche rafael_cleaning
    result = subprocess.run(
        ['git', 'checkout', '-b', 'rafael_cleaning', 'origin/rafael_cleaning'],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        # Si la branche locale existe déjà, juste switcher
        subprocess.run(['git', 'checkout', 'rafael_cleaning'], capture_output=True)
    
    # Installation du package en mode éditable
    print("📦 Installation du package...")
    result = subprocess.run(['pip', 'install', '-e', '.', '--quiet'], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ Erreur installation: {result.stderr}")
    else:
        print("✅ Package installé")
    
    print("💾 Montage Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Extraction dataset
    archive_data = '/content/drive/MyDrive/DS_COVID/archive_covid.zip'
    if os.path.exists(archive_data):
        print("📦 Extraction dataset...")
        os.makedirs('./data/raw/', exist_ok=True)
        subprocess.run(['unzip', '-o', '-q', archive_data, '-d', './data/raw/COVID-19_Radiography_Dataset/'])
    
    print("✅ Bootstrap terminé")


# =============================================================================
# CONFIGURATION DES CHEMINS
# =============================================================================

# Déterminer project_root selon l'environnement
if ENV == "colab":
    project_root = Path('/content/Data_Pipeline')
elif ENV == "wsl":
    project_root = Path('/home/cepa/DST/projet_DS/Data_Pipeline/Data_Pipeline')
else:  # local
    project_root = Path.cwd().parent.parent

# Charger la configuration depuis JSON
from src.utils.config import build_config

config = build_config(project_root, ENV)

# Exports pour compatibilité avec anciens notebooks
data_dir = config.data_dir
categories = config.classes
img_size = config.img_size


# =============================================================================
# IMPORTS ML/DL
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

# =============================================================================
# CONFIGURATION MATPLOTLIB
# =============================================================================

plt.rcParams['figure.figsize'] = (15, 10)
sns.set_style('whitegrid')

# =============================================================================
# AFFICHAGE DU RÉSUMÉ
# =============================================================================

print("\n" + "=" * 70)
print("✅ CONFIGURATION PRÊTE - Data Pipeline")
print("=" * 70)
print(f"📂 Projet: {project_root}")
print(f"📊 Dataset: {data_dir}")
print(f"🏷️ Classes: {', '.join(categories)}")
print(f"🔧 Batch: {config.batch_size} | Époques: {config.epochs}")
print(f"📐 Dataset accessible: {'✅' if data_dir.exists() else '❌'}")
print("=" * 70)

In [ ]:
# Imports des fonctions utilitaires
from src.notebooks import (
    load_dataset,
    create_preprocessing_pipeline,
    prepare_train_val_test_split,
    compute_class_weights,
    build_transfer_learning_model,
    create_transfer_learning_generators,
    compile_model,
    create_callbacks,
    unfreeze_top_layers,
    train_model,
    evaluate_model,
    plot_training_curves,
    plot_confusion_matrix,
    select_sample_images,
    run_gradcam_analysis,
)

print("✅ Fonctions utilitaires importées")

## 2. Chargement et Préparation des Données

In [ ]:
# Charger le dataset
image_paths, mask_paths, labels, labels_int = load_dataset(
    data_dir=config.data_dir,
    categories=config.classes,
    n_images_per_class=None,
    load_masks=False,
    verbose=True
)

In [ ]:
# ⚠️ IMPORTANT: Transfer Learning nécessite 224x224 (pas 128x128)
pipeline = create_preprocessing_pipeline(
    img_size=(224, 224),  # ⭐ 224x224 pour Transfer Learning
    color_mode='RGB',
    mask_paths=None,
    verbose=True
)

print("\nChargement des images...")
images = pipeline.fit_transform(image_paths)
images = images.astype('float32') / 255.0

print(f"\n📊 Images préparées:")
print(f"  Shape: {images.shape}")
print(f"  Range: [{images.min():.3f}, {images.max():.3f}]")
print(f"  Dtype: {images.dtype}")

In [ ]:
# Split train/val/test
X_train, X_val, X_test, y_train_cat, y_val_cat, y_test_cat = prepare_train_val_test_split(
    images=images,
    labels_int=labels_int,
    num_classes=len(config.classes),
    test_size=0.15,
    val_size=0.15,
    random_seed=config.random_seed,
    verbose=True
)

# Récupérer les labels integer
y_train = np.argmax(y_train_cat, axis=1)
y_val = np.argmax(y_val_cat, axis=1)
y_test = np.argmax(y_test_cat, axis=1)

In [ ]:
# Calculer les class weights
class_weights = compute_class_weights(
    y_train=y_train,
    categories=config.classes,
    verbose=True
)

## 3. Transfer Learning - InceptionV3

Nous allons utiliser InceptionV3 avec une stratégie en 2 phases.

### Construction du Modèle

In [ ]:
# Créer le modèle Transfer Learning
model, base_model = build_transfer_learning_model(
    base_model_name='InceptionV3',
    input_shape=(224, 224, 3),
    num_classes=len(config.classes),
    freeze_base=True,  # Phase 1: Feature Extraction
    dropout_rate=0.3,
    dense_units=128,
    l2_reg=0.01,
    verbose=True
)

# Afficher le résumé
model.summary()

### Data Generators avec Preprocessing InceptionV3

In [ ]:
# Créer les générateurs avec preprocessing InceptionV3 (normalisation [-1, 1])
train_generator, val_generator, test_generator = create_transfer_learning_generators(
    X_train=X_train,
    y_train_cat=y_train_cat,
    X_val=X_val,
    y_val_cat=y_val_cat,
    X_test=X_test,
    y_test_cat=y_test_cat,
    base_model_name='InceptionV3',
    batch_size=32,
    augment_train=True,
    verbose=True
)

### Phase 1: Feature Extraction

Base model gelée, entraînement uniquement du head (classification).

In [ ]:
# Compiler le modèle
model = compile_model(
    model=model,
    learning_rate=0.001,
    verbose=True
)

In [ ]:
# Créer les callbacks
callbacks_fe = create_callbacks(
    models_dir=config.results_dir / 'transfer_learning_models' / 'inceptionv3_fe',
    monitor='val_accuracy',
    patience_early_stop=15,
    patience_reduce_lr=5,
    verbose=True
)

In [ ]:
# PHASE 1: Feature Extraction
print("="*70)
print("🚀 PHASE 1: FEATURE EXTRACTION")
print("="*70)

EPOCHS_FE = 80

history_fe = train_model(
    model=model,
    train_generator=train_generator,
    val_generator=val_generator,
    class_weights=class_weights,
    epochs=EPOCHS_FE,
    callbacks=callbacks_fe,
    verbose=True
)

### Phase 2: Fine-Tuning

Dégeler les 30 dernières couches d'InceptionV3 pour un fine-tuning.

In [ ]:
# Dégeler les top layers
model = unfreeze_top_layers(
    base_model=base_model,
    model=model,
    n_layers=30,  # InceptionV3: beaucoup de layers
    learning_rate=5e-5,  # 100x plus faible!
    verbose=True
)

In [ ]:
# Callbacks pour fine-tuning
callbacks_ft = create_callbacks(
    models_dir=config.results_dir / 'transfer_learning_models' / 'inceptionv3_ft',
    monitor='val_accuracy',
    patience_early_stop=15,
    patience_reduce_lr=5,
    verbose=True
)

In [ ]:
# PHASE 2: Fine-Tuning
print("="*70)
print("🚀 PHASE 2: FINE-TUNING")
print("="*70)

EPOCHS_FT = 50

history_ft = train_model(
    model=model,
    train_generator=train_generator,
    val_generator=val_generator,
    class_weights=class_weights,
    epochs=EPOCHS_FT,
    callbacks=callbacks_ft,
    verbose=True
)

## 4. Visualisation des Courbes d'Apprentissage

In [ ]:
# Plot training curves pour Fine-Tuning
plots_dir = config.results_dir / 'transfer_learning_plots'
plots_dir.mkdir(parents=True, exist_ok=True)

fig = plot_training_curves(
    history=history_ft,
    save_path=plots_dir / 'training_curves_ft.png',
    figsize=(15, 12)
)
plt.show()

## 5. Évaluation

In [ ]:
# Évaluer le modèle
y_pred, y_pred_proba = evaluate_model(
    model=model,
    X_test=X_test,
    y_test_cat=y_test_cat,
    y_test=y_test,
    categories=config.classes,
    verbose=True
)

In [ ]:
# Matrice de confusion
fig = plot_confusion_matrix(
    y_test=y_test,
    y_pred=y_pred,
    categories=config.classes,
    save_path=plots_dir / 'confusion_matrix.png',
    figsize=(10, 8)
)
plt.show()

## 6. Interprétabilité - Grad-CAM

In [ ]:
# Sélectionner des images échantillons
sample_indices = select_sample_images(
    X_test=X_test,
    y_test=y_test,
    y_pred=y_pred,
    y_pred_proba=y_pred_proba,
    categories=config.classes,
    n_samples=6,
    random_seed=42,
    verbose=True
)

In [ ]:
# Analyse Grad-CAM
interp_dir = config.results_dir / 'interpretability_transfer_learning'
interp_dir.mkdir(parents=True, exist_ok=True)

gradcam, heatmaps = run_gradcam_analysis(
    model=model,
    X_test=X_test,
    y_pred=y_pred,
    y_pred_proba=y_pred_proba,
    categories=config.classes,
    sample_indices=sample_indices,
    save_dir=interp_dir,
    verbose=True
)

plt.show()

## 7. Résumé

✅ Transfer Learning avec InceptionV3 - 2 phases

**Avantages du Transfer Learning:**
- Convergence plus rapide
- Meilleure généralisation
- Moins de données nécessaires
- Poids ImageNet pré-entraînés

**Stratégie 2-phases:**
1. **Feature Extraction** (80 epochs) - Base gelée
2. **Fine-Tuning** (50 epochs) - Top layers dégelées

**Code ultra-compact:**
- 3 fonctions pour créer et entraîner le modèle
- Preprocessing automatique selon le modèle
- Compatible avec VGG16, ResNet50, EfficientNetB0, InceptionV3

In [ ]:
print("="*70)
print("🎉 NOTEBOOK TERMINÉ")
print("="*70)
print("\n✅ Modèle Transfer Learning entraîné (2 phases)")
print("✅ Visualisations générées")
print("✅ Interprétabilité analysée")
print("\n💡 Pour tester un autre modèle:")
print("   - Changez 'InceptionV3' par 'VGG16', 'ResNet50', ou 'EfficientNetB0'")
print("   - Relancez depuis la section 3")